In [5]:
import pandas as pd

def clean_sales_data(file_path, output_path):
    print("Initializing Professional Analytics Data Pipeline...")

    # 1. Load the raw transactional dataset
    df = pd.read_csv(file_path)

    # 2. Clean 'Amount' column (Strip formatting strings and cast to float)
    # Professional standard: Use regex=False for simple replacements to maximize speed
    df['Amount'] = df['Amount'].str.replace('$', '', regex=False)
    df['Amount'] = df['Amount'].str.replace(',', '', regex=False)
    df['Amount'] = df['Amount'].astype(float)

    # 3. Explicitly parse Dates to datetime objects
    # Dayfirst=True ensures international DD/MM/YYYY standards are respected
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

    # 4. Feature Engineering: Inject Business Value Metrics
    # Calculating unit values allows us to build pricing-strategy analyses later
    df['Price_Per_Box'] = (df['Amount'] / df['Boxes Shipped']).round(2)

    # Extract structural time attributes for optimized grouping performance
    df['Year'] = df['Date'].dt.year
    df['Month_Name'] = df['Date'].dt.strftime('%B')
    df['Quarter'] = 'Q' + df['Date'].dt.quarter.astype(str)

    # 5. Data Integrity Sanity Check
    # Ensure no sales records dropped below a logical threshold ($0.00)
    anomalies = df[df['Amount'] <= 0]
    if not anomalies.empty:
        print(f" Warning: Found {len(anomalies)} rows with non-positive sales amount. Filtering out.")
        df = df[df['Amount'] > 0]
    else:
        print(" Data integrity check passed: No negative or zero-dollar sales found.")

    # 6. Export optimized asset
    df.to_csv(output_path, index=False)
    print(f" Cleaned dataset successfully compiled and exported to: '{output_path}'")

    # Return quick summary overview
    print("\n Data Pipeline Schema Summary:")
    print(df.dtypes)
    return df

# Execute the pipeline
cleaned_df = clean_sales_data('Chocolate Sales (2).csv', 'Cleaned_Chocolate_Sales.csv')

Initializing Professional Analytics Data Pipeline...
 Data integrity check passed: No negative or zero-dollar sales found.
 Cleaned dataset successfully compiled and exported to: 'Cleaned_Chocolate_Sales.csv'

 Data Pipeline Schema Summary:
Sales Person             object
Country                  object
Product                  object
Date             datetime64[ns]
Amount                  float64
Boxes Shipped             int64
Price_Per_Box           float64
Year                      int32
Month_Name               object
Quarter                  object
dtype: object
